In [3]:
import torch
import torch.nn as nn
import json
import re

device = torch.device("cpu")

with open("vocabulary.json","r",encoding = "utf-8") as file:
    vocab = json.load(file)

with open("config.json","r",encoding = 'utf-8') as file:
    config = json.load(file)

MAX_LENGTH = config["max_length"]
EMBEDDING_DIM = config["embedding_dim"]

print("vocabulary size : ",len(vocab))
print("Maximum sequence length : ",MAX_LENGTH)
print("Embedding Dimension: ",EMBEDDING_DIM)
print("Device: ",device)

vocabulary size :  20000
Maximum sequence length :  200
Embedding Dimension:  64
Device:  cpu


In [4]:
class SentimentModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=64
    ):
        super().__init__()

        self.embedding = nn.Embedding(
        vocab_size,
        embedding_dim,
        padding_idx=0
        )

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(
        embedding_dim,
        1
        )
    def forward(self,input_ids):
        embedded = self.embedding(input_ids)

        mask = (input_ids != 0).unsqueeze(-1)

        masked = embedded * mask

        summed = masked.sum(dim=1)

        counts = mask.sum(dim=1).clamp(min=1)

        pooled = summed/counts
        pooled = self.dropout(pooled)

        logits = self.classifier(pooled)

        return logits.squeeze(1)

In [5]:
model = SentimentModel(
    vocab_size = len(vocab),
    embedding_dim=EMBEDDING_DIM
    ).to(device)

state = torch.load(
    "sentiment_model.pth",
    map_location=device,
    weights_only=True
)

model.load_state_dict(state)
model.eval()

print("Trained model loaded!!")

Trained model loaded!!


In [7]:
def tokenize(text):
    return re.findall(r"[a-z0-9']+",text.lower())

In [8]:
def encode_text(text):
    tokens = tokenize(text)

    ids = [
    vocab.get(token,vocab["<UNK>"])
    for token in tokens
    ]

    ids = ids[:MAX_LENGTH]

    ids += [vocab["<PAD>"]] * (MAX_LENGTH - len(ids))

    return ids

In [9]:
def predict_sentiment(text):
    model.eval()

    input_ids = torch.tensor(
    encode_text(text),
    dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        logit=model(input_ids)
        probability=torch.sigmoid(logit).item()

    sentiment = (
    "Positive" if probability >= 0.5
    else "Negative"
    )
    return sentiment , probability

In [10]:
reviews = [
    "The movie was absolutely fantastic. The acting, direction, and storyline were amazing.",

    "This film was a complete disappointment. The story was boring and the acting was terrible.",

    "I loved every minute of this movie. The characters were interesting and the ending was perfect.",

    "What a waste of time! The screenplay was weak, the dialogues were awful, and the movie was too long.",

    "The cinematography was beautiful and the music was excellent, but the story was average.",

    "The actors delivered outstanding performances. This is one of the best movies I have ever watched.",

    "I expected something great, but the movie was predictable, boring, and poorly executed.",

    "The first half was entertaining, but the second half was disappointing and confusing.",

    "An emotional masterpiece with brilliant performances, beautiful music, and a powerful storyline.",

    "The movie had good visuals and talented actors, but the weak script made it difficult to enjoy."
]

In [11]:
for review in reviews:
    sentiment,probability = predict_sentiment(review)

    print("Review: ",review)
    print("Prediction: ",sentiment)
    print(f"Positive probability: {probability:.4f}")
    print("-"*60)

Review:  The movie was absolutely fantastic. The acting, direction, and storyline were amazing.
Prediction:  Positive
Positive probability: 0.9998
------------------------------------------------------------
Review:  This film was a complete disappointment. The story was boring and the acting was terrible.
Prediction:  Negative
Positive probability: 0.0000
------------------------------------------------------------
Review:  I loved every minute of this movie. The characters were interesting and the ending was perfect.
Prediction:  Positive
Positive probability: 0.9989
------------------------------------------------------------
Review:  What a waste of time! The screenplay was weak, the dialogues were awful, and the movie was too long.
Prediction:  Negative
Positive probability: 0.0000
------------------------------------------------------------
Review:  The cinematography was beautiful and the music was excellent, but the story was average.
Prediction:  Positive
Positive probability: